# CLIPZyme+ Cofactor Prediction Pipeline

Standalone notebook that predicts enzyme **cofactors** from structures using
[`CLIPZymePlus`](clipzyme/lightning/clipzyme_plus.py). Given a dataset CSV with
`reaction`, `sequence`, `protein_id`, `cif` columns (see the CLIPZyme README):

1. (optional) downloads predicted structures from the AlphaFold DB
2. builds a `ReactionDataset` and runs `CLIPZymePlus` to embed each enzyme with
   the CLIPZyme protein encoder and classify its cofactor with the MLP ensemble
3. writes one CSV with the original rows plus `predicted_cofactor`, the top-k
   classes, and per-class probabilities.

Only the enzyme **structures** are needed for cofactor prediction, so the
`reaction` column may be left empty. The CLIPZyme reaction–enzyme screening
*score* is a separate quantity — see the "Screening with CLIPZyme" section of the
README to compute it.

## 1. Imports

In [ ]:
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from clipzyme import CLIPZymePlus, ReactionDataset
from clipzyme.utils.loading import ignore_None_collate

## 2. Configuration

Edit the paths below before running. `DATASET_CSV` must contain at least
`reaction`, `sequence`, `protein_id`, `cif` columns. Download the checkpoints
from Zenodo into `files/` first:

```bash
wget https://zenodo.org/records/11187747/files/clipzyme_model.zip && unzip clipzyme_model.zip -d files
wget https://zenodo.org/records/20673359/files/clipzyme_plus_cofactor_ensemble.pt -P files/
```

In [ ]:
REPO_ROOT = Path(".").resolve()

# --- Inputs ---------------------------------------------------------------
DATASET_CSV = REPO_ROOT / "files" / "new_data.csv"            # reaction dataset
OUTPUT_CSV  = REPO_ROOT / "clipzyme_cofactor_predictions.csv"

# --- Checkpoints ----------------------------------------------------------
CHECKPOINT_PATH          = REPO_ROOT / "files" / "clipzyme_model.ckpt"
COFACTOR_CHECKPOINT_PATH = REPO_ROOT / "files" / "clipzyme_plus_cofactor_ensemble.pt"
ESM_DIR                  = Path("/path/to/esm2_dir")          # dir holding esm2_t33_650M_UR50D.pt

# --- Optional protein-graph cache (speeds up re-runs) ---------------------
PROTEIN_CACHE_DIR = None                                      # e.g. REPO_ROOT / "files" / "protein_cache"

# --- AlphaFold DB download (optional) -------------------------------------
DOWNLOAD_AF_STRUCTURES = False                                # set True to fetch CIFs by UniProt id
AF_OUTPUT_DIR          = REPO_ROOT / "files" / "af2_cifs"

# --- Inference ------------------------------------------------------------
DEVICE              = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE          = 8
NUM_WORKERS         = 4
TOP_K               = 5
EXCLUDE_NO_COFACTOR = False

## 3. Load reaction dataset

In [ ]:
df = pd.read_csv(DATASET_CSV)
df["protein_id"] = df["protein_id"].astype(str)

required = {"reaction", "sequence", "protein_id", "cif"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

print(f"Loaded {len(df)} rows / {df['protein_id'].nunique()} unique proteins from {DATASET_CSV}")
df.head()

## 4. (Optional) Download AlphaFold DB structures

Uses `gsutil` to fetch `gs://public-datasets-deepmind-alphafold-v4/AF-{uniprot}-F1-model_v4.cif`
for every unique `protein_id`, then rewrites the `cif` column to point at the
downloaded files. Requires [`gsutil`](https://cloud.google.com/storage/docs/gsutil_install) on `PATH`.

In [ ]:
if DOWNLOAD_AF_STRUCTURES:
    AF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    uniprot_ids = sorted(df["protein_id"].unique().tolist())
    needed = [u for u in uniprot_ids if not (AF_OUTPUT_DIR / f"AF-{u}-F1-model_v4.cif").exists()]
    print(f"{len(uniprot_ids)} unique IDs; {len(needed)} need downloading.")
    if needed:
        gs_paths = [f"gs://public-datasets-deepmind-alphafold-v4/AF-{u}-F1-model_v4.cif" for u in needed]
        paths_file = AF_OUTPUT_DIR / "uniprot_cif_paths.txt"
        paths_file.write_text("\n".join(gs_paths))
        subprocess.run(f"cat {paths_file} | gsutil -m cp -I {AF_OUTPUT_DIR}/", shell=True, check=True)
    df["cif"] = df["protein_id"].apply(lambda u: str(AF_OUTPUT_DIR / f"AF-{u}-F1-model_v4.cif"))
else:
    print("Skipping AlphaFold download — using `cif` paths from the input CSV.")

## 5. Validate CIF paths

Rows whose CIF is missing (or whose sequence is empty / longer than 1000 aa) are
dropped silently by `ReactionDataset.skip_sample`, so we report missing CIFs up front.

In [ ]:
missing_cif_mask = ~df["cif"].apply(lambda p: Path(str(p)).exists())
n_missing = int(missing_cif_mask.sum())
if n_missing:
    print(f"WARNING: {n_missing} / {len(df)} rows reference a CIF that does not exist; these will be skipped.")
    display(df.loc[missing_cif_mask, ["protein_id", "cif"]].head())
else:
    print("All CIF paths exist on disk.")

# ReactionDataset reads from a CSV path, so persist any edits (e.g. AF download) first.
scratch_csv = OUTPUT_CSV.parent / f"{DATASET_CSV.stem}__resolved.csv"
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(scratch_csv, index=False)
print(f"Wrote resolved dataset to {scratch_csv}")

## 6. Build the dataset and load CLIPZyme+

`use_as_protein_encoder=True` tells `ReactionDataset` to ignore reactions during
filtering — only the protein structures matter for cofactor prediction. The
cofactor ensemble is loaded from `COFACTOR_CHECKPOINT_PATH` (auto-downloaded from
Zenodo if missing).

In [ ]:
dataset = ReactionDataset(
    dataset_file_path=str(scratch_csv),
    esm_dir=str(ESM_DIR),
    protein_cache_dir=str(PROTEIN_CACHE_DIR) if PROTEIN_CACHE_DIR else None,
    use_as_protein_encoder=True,
)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    collate_fn=ignore_None_collate,
)

model = CLIPZymePlus(
    checkpoint_path=str(CHECKPOINT_PATH),
    cofactor_checkpoint_path=str(COFACTOR_CHECKPOINT_PATH),
    device=DEVICE,
)
model = model.eval()
print(f"Loaded CLIPZyme+ with a {len(model.cofactor_members)}-member ensemble "
      f"over {len(model.cofactor_class_names)} cofactor classes.")

## 7. Predict cofactors

Each batch is embedded by the CLIPZyme protein encoder and classified by the MLP
ensemble. `CofactorOutput` gives the top-1 class, per-class ensemble-mean
probabilities, and `model.top_k(...)` for ranked predictions.

In [ ]:
records = []
for batch in tqdm(loader, desc="Cofactor prediction"):
    if batch is None:            # whole batch was filtered out by ignore_None_collate
        continue
    output = model(batch)
    ranked = model.top_k(output, k=TOP_K, exclude_no_cofactor=EXCLUDE_NO_COFACTOR)
    for i, pid in enumerate(output.sample_ids):
        top = ranked[i]
        rec = {
            "protein_id": str(pid),
            "predicted_cofactor": top[0]["class_name"],
            "predicted_cofactor_prob": top[0]["probability_mean"],
            "ensemble_size": len(model.cofactor_members),
        }
        for r in top:
            rec[f"top{r['rank']}_class"]     = r["class_name"]
            rec[f"top{r['rank']}_prob_mean"] = r["probability_mean"]
            rec[f"top{r['rank']}_prob_std"]  = r["probability_std"]
        for ci, cname in enumerate(output.class_names):
            rec[f"prob__{cname}"] = float(output.probabilities[i, ci])
        records.append(rec)

cofactor_df = pd.DataFrame(records)
print(f"Cofactor predictions for {len(cofactor_df)} proteins.")
cofactor_df.head()

## 8. Merge & save final CSV

In [ ]:
merged = df.merge(cofactor_df, on="protein_id", how="left")
merged.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(merged)} rows to {OUTPUT_CSV}")
merged.head()